In [1]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 63.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 70.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.4/188.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/metadata-csv/metadata.csv
/kaggle/input/merged-dataset/metadata_with_audiolinks.csv
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/README.md
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/.gitattributes
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191011_221500.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20200331_011500.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20190911_233000.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20200215_230000.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191225_210000.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20200207_040000.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20190920_004500.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191021_183000.wav
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT209

In [3]:
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import soundfile as sf
import wave
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import gc # For deleting in-memory stuff

import chromadb
from chromadb.config import Settings
from chromadb import PersistentClient

## Load pre-trained model

In [4]:
# Instantiate model with pre-trained weights
perch_model = hub.KerasLayer(
    "https://www.kaggle.com/models/google/bird-vocalization-classifier/frameworks/TensorFlow2/variations/bird-vocalization-classifier/versions/8",
    signature='serving_default',
    trainable=False,
    output_key="embedding"
)

## Load metadata file with links to audio files

In [4]:
# Load metadata.csv
metadata_withlinks_path = '/kaggle/input/merged-dataset/metadata_with_audiolinks.csv'
metadata_merged_df = pd.read_csv(metadata_withlinks_path)
metadata_merged_df.head()

,sample_name,fname,min_t,max_t,site,date,species_number,subset,SPHSUR,BOABIS,...,ADEMAR,BOAALM,PHYDIS,RHIORN,LEPFLA,SCIRIZ,DENELE,SCIALT,filename,file_path
0,SAMPLE_00000.wav,INCT20955_20190904_003000,0,3,INCT20955,2019-09-04 00:30:00,4,test,1,1,...,0,0,0,0,0,0,0,0,inct20955_20190904_003000.wav,/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_...
1,SAMPLE_00001.wav,INCT20955_20190904_003000,1,4,INCT20955,2019-09-04 00:30:00,4,test,1,1,...,0,0,0,0,0,0,0,0,inct20955_20190904_003000.wav,/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_...
2,SAMPLE_00002.wav,INCT20955_20190904_003000,2,5,INCT20955,2019-09-04 00:30:00,4,test,1,1,...,0,0,0,0,0,0,0,0,inct20955_20190904_003000.wav,/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_...
3,SAMPLE_00003.wav,INCT20955_20190904_003000,3,6,INCT20955,2019-09-04 00:30:00,4,test,1,1,...,0,0,0,0,0,0,0,0,inct20955_20190904_003000.wav,/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_...
4,SAMPLE_00004.wav,INCT20955_20190904_003000,4,7,INCT20955,2019-09-04 00:30:00,4,test,1,1,...,0,0,0,0,0,0,0,0,inct20955_20190904_003000.wav,/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_...


In [5]:
metadata_merged_df['file_path'].iloc[0]

'/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20190904_003000.wav'

In [6]:
metadata_merged_df['file_path'].value_counts()

file_path
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20190904_003000.wav    58
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT17/INCT17_20200311_221500.wav          58
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT17/INCT17_20200313_223000.wav          58
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT17/INCT17_20200313_213000.wav          58
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT17/INCT17_20200313_203000.wav          58
                                                                                             ..
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191111_204500.wav    42
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191111_181500.wav    42
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191107_000000.wav    41
/kaggle/input/anuraset-v2-raw/AnuraSet_v2.0.0_raw/INCT20955/INCT20955_20191107_010000.wav    28
/kaggle/input/anuraset-v2-raw/

In [7]:
metadata_merged_df.shape

(93378, 52)

## Pre-process the audio so we feed it into Perch for embeddings

In [8]:
# Make a directory for the files
processed_audio_dir = '/kaggle/working/processed_audio'
os.makedirs(processed_audio_dir, exist_ok=True)

In [9]:
# Instead of processing row-by-row
file_groups = metadata_merged_df.groupby('file_path')

for file_path, group in file_groups:
    full_audio = librosa.load(file_path, sr=32000)[0]
    for _, row in group.iterrows():
        start = int(row['min_t'] * 32000)
        end = int(row['max_t'] * 32000)
        segment = full_audio[start:end]
        # Process and save segment

In [10]:
def process_grouped_files(metadata_df, output_dir):
    """Process grouped files with memory efficiency"""
    os.makedirs(output_dir, exist_ok=True)
    
    for file_path, group in metadata_df.groupby('file_path'):
        # Load full audio ONCE per source file
        full_audio, sr = librosa.load(file_path, sr=32000, mono=True)
        full_audio = full_audio.astype(np.float32)  # Convert early to save memory

        for _, row in group.iterrows():
            # Calculate sample positions
            start = int(row['min_t'] * sr)
            end = int(row['max_t'] * sr)
            
            # Extract segment
            segment = full_audio[start:end].copy()  # Copy avoids memory view issues
            
            # Process using your existing logic
            processed = process_segment(segment, row, sr)
            
            # Save immediately
            save_path = os.path.join(output_dir, f"{row['sample_name']}.npy")
            np.save(save_path, processed)
            
            # Clean up
            del segment, processed
            gc.collect()

In [11]:
def process_segment(segment, row, sr):
    """Modified preprocessing for extracted segments"""
    target_samples = int(sr * (row['max_t'] - row['min_t']))
    
    if len(segment) < target_samples:
        segment = np.pad(segment, (0, target_samples - len(segment)))
    else:
        segment = segment[:target_samples]
    
    return segment

In [12]:
dfs = np.array_split(metadata_merged_df, 4)
first_df, second_df, third_df, fourth_df = dfs

/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [13]:
# process_grouped_files(metadata_merged_df, processed_audio_dir)
process_grouped_files(first_df, processed_audio_dir)
# process_grouped_files(second_half_df, processed_audio_dir)

# # Verify output
# test_file = "/kaggle/working/processed_audio/SAMPLE_00000.npy"
# print(np.load(test_file).shape)

In [14]:
test_file = "/kaggle/working/processed_audio/SAMPLE_26368.wav.npy"
print(np.load(test_file).shape)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/processed_audio/SAMPLE_26368.wav.npy'

In [15]:
# Check that files were processes
processed_files = os.listdir(processed_audio_dir)
print(f"Total files in processed_audio_dir: {len(processed_files)}")

Total files in processed_audio_dir: 23345


In [16]:
os.listdir(processed_audio_dir)[:5]

['SAMPLE_13303.wav.npy',
 'SAMPLE_11462.wav.npy',
 'SAMPLE_14546.wav.npy',
 'SAMPLE_14797.wav.npy',
 'SAMPLE_13165.wav.npy']

In [17]:
!zip -r /kaggle/working/processed_audio.zip /kaggle/working/processed_audio

  adding: kaggle/working/processed_audio/ (stored 0%)
  adding: kaggle/working/processed_audio/SAMPLE_13303.wav.npy (deflated 9%)
  adding: kaggle/working/processed_audio/SAMPLE_11462.wav.npy (deflated 10%)
  adding: kaggle/working/processed_audio/SAMPLE_14546.wav.npy (deflated 8%)
  adding: kaggle/working/processed_audio/SAMPLE_14797.wav.npy (deflated 12%)
  adding: kaggle/working/processed_audio/SAMPLE_13165.wav.npy (deflated 14%)
  adding: kaggle/working/processed_audio/SAMPLE_09352.wav.npy (deflated 14%)
  adding: kaggle/working/processed_audio/SAMPLE_05482.wav.npy (deflated 10%)
  adding: kaggle/working/processed_audio/SAMPLE_05480.wav.npy (deflated 10%)
  adding: kaggle/working/processed_audio/SAMPLE_19053.wav.npy (deflated 10%)
  adding: kaggle/working/processed_audio/SAMPLE_00396.wav.npy (deflated 9%)
  adding: kaggle/working/processed_audio/SAMPLE_07638.wav.npy (deflated 9%)
  adding: kaggle/working/processed_audio/SAMPLE_13842.wav.npy (deflated 12%)
  adding: kaggle/working/p

In [ ]:
# Initialize ChromaDB client with persistent storage
client = PersistentClient(path="chroma_db_storage/")

# Create or get the collection
collection = client.get_or_create_collection(name="embeddings")
print(client.heartbeat())

In [ ]:
# client = chromadb.Client()
# collection = client.create_collection(name="embeddings")

In [ ]:
# embeddings_dir = '/kaggle/working/embeddings'
# os.makedirs(embeddings_dir, exist_ok=True)

In [ ]:
# practice_collection = client.create_collection("practice")

In [ ]:
# practice_collection.add(
#     embeddings=[
#         [1.1, 2.3, 3.2],
#         [4.5, 6.9, 4.4],
#         [1.1, 2.3, 3.2],
#         [4.5, 6.9, 4.4],
#         [1.1, 2.3, 3.2],
#         [4.5, 6.9, 4.4],
#         [1.1, 2.3, 3.2],
#         [4.5, 6.9, 4.4],
#     ],
#     metadatas=[
#         {"uri": "img1.png", "style": "style1"},
#         {"uri": "img2.png", "style": "style2"},
#         {"uri": "img3.png", "style": "style1"},
#         {"uri": "img4.png", "style": "style1"},
#         {"uri": "img5.png", "style": "style1"},
#         {"uri": "img6.png", "style": "style1"},
#         {"uri": "img7.png", "style": "style1"},
#         {"uri": "img8.png", "style": "style1"},
#     ],
#     documents=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6", "doc7", "doc8"],
#     ids=["id1", "id2", "id3", "id4", "id5", "id6", "id7", "id8"],
# )

# query_result = practice_collection.query(
#         query_embeddings=[[1.1, 2.3, 3.2], [5.1, 4.3, 2.2]],
#         n_results=2,
#     )

# print(query_result)

In [ ]:
# def extract_embeddings(audio_files_dir, model):
#     """Stores audio embeddings in ChromaDB with proper data handling"""
#     ids = []
#     embeddings = []

#     # Get full paths for audio files
#     audio_files = [os.path.join(audio_files_dir, file) for file in os.listdir(audio_files_dir)]

#     for idx, file_path in enumerate(audio_files[:5]):
#         # Load and prepare audio
#         audio, sr = librosa.load(file_path, sr=32000)
#         audio_tensor = tf.convert_to_tensor(audio, dtype=tf.float32)
#         audio_batch = tf.expand_dims(audio_tensor, 0)

#         # Extract embedding
#         embedding = model(audio_batch).numpy().tolist()[0]

#         # Batch collection
#         ids.append(f"audio_{idx}")
#         embeddings.append(embedding)
        
#         # Bulk insert for efficiency
#     collection.add(
#         ids=ids,
#         embeddings=embeddings,
#         metadatas=[{"path": path} for path in audio_files[:5]]  # Add metadata
#     )
    
#     print(f"Stored {len(ids)} embeddings")
#     return collection.get()  # Returns first 5 items by default
            
# extract_embeddings(processed_audio_dir, perch_model)          

In [ ]:
def extract_embeddings(audio_files_dir, model):
    """Extract Perch embeddings from pre-processed audio"""
    ids = []
    embeddings = []
    metadatas = []

    # Get full paths for audio files
    audio_files = [os.path.join(audio_files_dir, file) for file in os.listdir(audio_files_dir)]

    for idx, file_path in enumerate(audio_files[:5]):  # Process only the first 5 files
        try:
            # Load and prepare audio
            audio, sr = librosa.load(file_path, sr=32000)
            audio_tensor = tf.convert_to_tensor(audio / np.max(np.abs(audio)), dtype=tf.float32)  # Normalize
            audio_batch = tf.expand_dims(audio_tensor, 0)  # Add batch dimension

            # Extract embedding
            embedding = model(audio_batch).numpy().tolist()[0]  # Convert to list

            # Collect data
            ids.append(f"audio_{idx}")
            embeddings.append(embedding)
            metadatas.append({"path": file_path})
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    # Debug embeddings before insertion into ChromaDB
    # print(f"Embeddings before insertion: {embeddings}")
    # print(f"Type of embeddings: {type(embeddings)}")
    # print(f"Type of first embedding: {type(embeddings[0])}")
    # print(f"Length of embeddings: {len(embeddings)}")

    return ids, embeddings, metadatas

In [ ]:
# Extract embeddings from audio files using the Perch model
ids, embeddings, metadatas = extract_embeddings(processed_audio_dir, perch_model)

In [ ]:
def save_embeddings(collection, ids, embeddings, metadatas):
    """Save Perch embeddings to ChromaDB"""
    # Ensure embeddings are valid before adding to ChromaDB
    if not all(len(lst) == len(ids) for lst in [embeddings, metadatas]):
        raise ValueError("IDs, embeddings, and metadata must have the same length")

    if len(embeddings) > 0 and all(isinstance(e, list) for e in embeddings):
        collection.upsert(
            ids=ids,
            embeddings=embeddings,
            metadatas=metadatas
        )
        print(f"Successfully stored {len(ids)} embeddings")
        return collection.get()  # Return stored data for verification
    else:
        print("No valid embeddings to save.")
        return None

In [ ]:
# Save the extracted embeddings to ChromaDB
saved_data = save_embeddings(collection, ids, embeddings, metadatas)

In [ ]:
result = collection.query(
    query_embeddings=[embeddings[0]],
    n_results=2
)
print(result)

In [ ]:
# Print the query results
print("Query Result IDs:", result['ids'])
print("Query Distances:", result['distances'])
print("Query Metadata:", result['metadatas'])

## Old stuff I may repurpose

In [ ]:
# from tqdm import tqdm  # For progress bar

# # Process audio files with a progress bar in the execution code
# results = []
# with tqdm(total=len(metadata_merged_df[:5]), desc="Processing audio files", unit="file") as pbar:
#     for idx, audio_file in enumerate(metadata_merged_df['file_path']):
#         try:
#             # Call the function to process audio
#             result = save_processed_audio(audio_file, processed_audio_dir)
#             results.append((audio_file, result))  # Store results for later use
#         except Exception as e:
#             print(f"Error processing {audio_file}: {e}")
        
#         # Update the progress bar
#         pbar.update(1)

In [ ]:
# def split_data(data_df, batch_size):
#     """Splits the df into batches of a given size."""
#     num_batches = (len(data_df) + batch_size - 1) // batch_size  # Ceiling division
#     batches = []
#     for i in range(num_batches):
#         start = i * batch_size
#         end = start + batch_size
#         batch = data_df.iloc[start:end].copy()
#         batch['batch_number'] = i + 1  # Add a batch number column
#         batches.append(batch)
#     return batches

# batch_size = 5000  # Number of files per batch
# batches = split_data(metadata_merged_df, batch_size)
# print(f"Number of batches: {len(batches)}")

In [ ]:
# print(f"Number of batches: {len(batches)}")
# for i, batch in enumerate(batches):
#     print(f"Batch {i+1} has {len(batch)} rows")

In [ ]:
# def extract_embeddings(audio_files_dir, model):
#     """Stores audio embeddings in ChromaDB with proper data handling"""
#     ids = []
#     embeddings = []

#     # Get full paths for audio files
#     audio_files = [os.path.join(audio_files_dir, file) for file in os.listdir(audio_files_dir)]

#     for idx, file_path in enumerate(audio_files[:5]):
#         try:
#             # Load and prepare audio
#             audio, sr = librosa.load(file_path, sr=32000)
#             audio_tensor = tf.convert_to_tensor(audio / np.max(np.abs(audio)), dtype=tf.float32)  # Normalize
#             audio_batch = tf.expand_dims(audio_tensor, 0)

#             # # Debug input shape and format
#             # print(f"Input shape for {file_path}: {audio_batch.shape}")

#             # Extract embedding and debug output
#             embedding = model(audio_batch)
#             print(f"Raw embedding output for {file_path}: {embedding}")

#             embedding = embedding.numpy().tolist()[0]  # Convert to list

#             # Batch collection
#             ids.append(f"audio_{idx}")
#             embeddings.append(embedding)
#         except Exception as e:
#             print(f"Error processing {file_path}: {e}")

#     # Debug embeddings before insertion into ChromaDB
#     # print(f"Embeddings before insertion: {embeddings}")
#     print(f"Type of embeddings: {type(embeddings)}")
#     print(f"Type of first embedding: {type(embeddings[0])}")
#     print(f"Length of embeddings: {len(embeddings)}")

#     # Ensure embeddings are valid before adding to ChromaDB
#     if len(embeddings) > 0 and all(isinstance(e, list) for e in embeddings):
#         collection.add(
#             ids=ids,
#             embeddings=embeddings,
#             metadatas=[{"path": path} for path in audio_files[:5]]  # Add metadata
#         )
#         print(f"Stored {len(ids)} embeddings")
#     else:
#         print("No valid embeddings were extracted.")

#     return collection.get()  # Returns first 5 items by default

# # Example usage
# extract_embeddings(processed_audio_dir, perch_model)

In [ ]:
# def extract_embeddings(preprocessed_dir, metadata_df, model, output_dir):
#     """Extracts embeddings from preprocessed audio files using the Perch model."""
    
#     embeddings = []

#     # Iterate over audio files
#     for idx, row in metadata_df.iterrows():
#         # Map raw file path to preprocessed directory
#         original_filename = os.path.basename(row['file_path'])  # e.g., INCT20955_20190904_003000.wav
#         audio_path = os.path.join(preprocessed_dir, original_filename)

#         if not os.path.exists(audio_path):
#             print(f"File not found: {audio_path}")
#             embeddings.append(None)
#             continue

#         # Load and prepare audio
#         audio, sr = librosa.load(audio_path, sr=32000)
#         audio = tf.convert_to_tensor(audio, dtype=tf.float32)
#         audio = tf.expand_dims(audio, 0)  # Add batch dimension

#         # Extract embedding
#         embedding = model(audio).numpy().flatten()

#         # Append embedding
#         embeddings.append(embedding)

#         # Save embedding as .npy file
#         embedding_file = os.path.join(output_dir, f"{original_filename}.npy")
#         np.save(embedding_file, embedding)

#         if idx % 10 == 0 or idx == len(metadata_df) - 1:
#             print(f"Extracted {idx + 1}/{len(metadata_df)} embeddings...")

#     # Add embeddings as a new column in the DataFrame
#     metadata_df = metadata_df.copy()  # Avoid SettingWithCopyWarning
#     metadata_df['embedding'] = embeddings

#     print(f"Extraction completed. Saved {len(embeddings)} embeddings.")
#     return metadata_df

In [ ]:
# # Check the filenames from the raw metadata
# print(metadata_merged_df['file_path'].head())
# print("\nRaw filenames extracted:")
# print(metadata_merged_df['file_path'].apply(os.path.basename).head())

In [ ]:
# if os.path.exists(processed_audio_dir):
#     processed_files = os.listdir(processed_audio_dir)
#     print(f"Total preprocessed files: {len(processed_files)}")
#     print("First 5 preprocessed files:", processed_files[:5])
# else:
#     print("Processed audio directory not found.")

In [ ]:
# processed_files = os.listdir('/kaggle/working/processed_audio')
# print("\nPreprocessed audio filenames:")
# print(processed_files[:5])  # Display the first 5 files

In [ ]:
# # Extract embeddings with filename matching
# metadata_with_embeddings = extract_embeddings(
#     preprocessed_dir=processed_audio_dir,
#     metadata_df=metadata_merged_df[:5].copy(),  # Use a copy to avoid warnings
#     model=perch_model,
#     output_dir=embeddings_dir
# )

In [ ]:
# # Check embeddings column
# metadata_merged_df.head()